# DRVI Threshold-Based Candidate Detection

Data-driven identification of candidate cell populations: instead of guessing a threshold, the distribution of a factor or gene (histogram + KDE) is checked for bimodality and the threshold is set in the valley between the two peaks. Covers both factor-based (DRVI dimensions) and gene-based variants, each with a matching candidate UMAP for visual inspection.

## 1. Factor threshold selection (step 1: candidate selector)

*Source: `2_7_a_drvi_threshold_selection.py`*

DRVI threshold selection — step 1: factor activity as a candidate selector

For each specified factor (+ direction), the distribution of factor values
across all cells is shown as a histogram + KDE. For a clean cell-type factor
the distribution is often bimodal: a mass near zero (cells without the
program) and a separated peak (the target cells). The threshold is set in a
data-driven way in the valley between the two peaks (local density minimum),
not guessed.

If no clear bimodality is found, the script falls back to a percentile and
explicitly marks it as not data-driven.

Result per factor direction:
  - Histogram plot with threshold line
  - candidate_cells_<factor><direction>.csv (cell barcodes above threshold)

Usage:
    conda run -n mapra_cytokines python 2_7_a_drvi_threshold_selection.py \
        --factor-dirs DR9-,DR17+,DR4+

In [ ]:
import argparse
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from scipy.signal import argrelextrema

In [ ]:
import argparse
args = argparse.Namespace(
    output_dir="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation",
    factor_dirs="DR9-,DR17+,DR4+,DR28+,DR21+",
    grid_points=2000,
    fallback_percentile=95.0,
    yscale="sqrt",
    celltype_key="cell_type_Scanorama",
    celltype_filter="",
    exclude="DR4+:Megakaryocytes",
    manual_threshold="DR4+:5,DR28+:4.2",
)

In [ ]:
embed_path = os.path.join(args.output_dir, "embed.h5ad")
embed = sc.read_h5ad(embed_path)


def _sanitize_celltype(name: str) -> str:
    return name.strip().replace(" ", "").replace("-", "").replace("_", "")


celltype_filter_list = [c.strip() for c in args.celltype_filter.split(",") if c.strip()]
if celltype_filter_list:
    keep = embed.obs[args.celltype_key].isin(celltype_filter_list).values
    print(f"Filter '{args.celltype_key}' in {celltype_filter_list}: {keep.sum()} / {embed.n_obs} cells")
    embed = embed[keep].copy()
    celltype_suffix = "_".join(_sanitize_celltype(c) for c in celltype_filter_list)
else:
    celltype_suffix = ""

exclude_map: dict[str, set[str]] = {}
for pair in [p.strip() for p in args.exclude.split(",") if p.strip()]:
    fd_key, celltype = pair.split(":")
    exclude_map.setdefault(fd_key.strip(), set()).add(celltype.strip())

manual_threshold_map: dict[str, float] = {}
for pair in [p.strip() for p in args.manual_threshold.split(",") if p.strip()]:
    fd_key, value = pair.split(":")
    manual_threshold_map[fd_key.strip()] = float(value)

factor_dirs = []
for fd in args.factor_dirs.split(","):
    fd = fd.strip()
    direction = fd[-1]
    factor = fd[:-1]
    assert direction in ("+", "-"), f"Could not read direction from '{fd}' (expected e.g. 'DR9-')."
    factor_dirs.append((factor, direction))

n = len(factor_dirs)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 4.5))
if n == 1:
    axes = [axes]

summary_rows = []

for ax, (factor, direction) in zip(axes, factor_dirs):
    fd_key = f"{factor}{direction}"
    excluded_types = exclude_map.get(fd_key, set())
    if excluded_types:
        keep_mask = ~embed.obs[args.celltype_key].isin(excluded_types).values
        n_excluded = int((~keep_mask).sum())
        print(f"{fd_key}: excluding {n_excluded} cells ({', '.join(sorted(excluded_types))})")
    else:
        keep_mask = np.ones(embed.n_obs, dtype=bool)

    obs_names_subset = embed.obs_names[keep_mask]
    values = np.asarray(embed[keep_mask, factor].X).flatten()
    grid = np.linspace(values.min(), values.max(), args.grid_points)
    kde = gaussian_kde(values)
    density = kde(grid)

    maxima_idx = argrelextrema(density, np.greater)[0]
    minima_idx = argrelextrema(density, np.less)[0]

    # Main ("bulk") peak: the overall highest density.
    bulk_idx = maxima_idx[np.argmax(density[maxima_idx])]
    bulk_x = grid[bulk_idx]

    # Target peak: highest density among the peaks that lie BEYOND the bulk
    # peak in the requested direction.
    if direction == "+":
        candidate_peaks = maxima_idx[grid[maxima_idx] > bulk_x]
    else:
        candidate_peaks = maxima_idx[grid[maxima_idx] < bulk_x]

    threshold = None
    is_data_driven = False
    is_manual = fd_key in manual_threshold_map

    if is_manual:
        threshold = manual_threshold_map[fd_key]
        is_data_driven = True
        print(f"{fd_key}: manual threshold={threshold} (overrides auto-detection)")
    elif len(candidate_peaks) > 0:
        target_idx = candidate_peaks[np.argmax(density[candidate_peaks])]
        target_x = grid[target_idx]
        # Valley between bulk_x and target_x: local minimum with the smallest density
        lo, hi = sorted([bulk_idx, target_idx])
        valley_candidates = minima_idx[(minima_idx > lo) & (minima_idx < hi)]
        if len(valley_candidates) > 0:
            valley_idx = valley_candidates[np.argmin(density[valley_candidates])]
            threshold = grid[valley_idx]
            is_data_driven = True

    if threshold is None:
        # Fallback: percentile in the requested direction, clearly marked
        # as not data-driven.
        pct = args.fallback_percentile if direction == "+" else (100 - args.fallback_percentile)
        threshold = np.percentile(values, pct)

    if direction == "+":
        candidate_mask = values > threshold
    else:
        candidate_mask = values < threshold
    n_candidates = int(candidate_mask.sum())

    candidates = pd.Series(obs_names_subset[candidate_mask], name="cell_barcode")
    ct_suffix = f"_in_{celltype_suffix}" if celltype_suffix else ""
    out_csv = os.path.join(args.output_dir, f"candidate_cells_{factor}{direction.replace('+','pos').replace('-','neg')}{ct_suffix}.csv")
    candidates.to_csv(out_csv, index=False)

    if is_manual:
        method = "manually set"
    elif is_data_driven:
        method = "data-driven (valley between peaks)"
    else:
        method = f"fallback: {args.fallback_percentile:.0f}th percentile"
    pct_of_total = 100 * n_candidates / embed.n_obs
    print(f"{factor}{direction}: threshold={threshold:.3f}  n_candidates={n_candidates} "
          f"({pct_of_total:.2f}% of all {embed.n_obs} cells)  [{method}]")
    print(f"  Saved: {os.path.basename(out_csv)}")

    summary_rows.append({
        "factor": factor, "direction": direction, "threshold": threshold,
        "n_candidates": n_candidates, "pct_candidates": pct_of_total,
        "n_excluded": len(embed) - len(values), "data_driven": is_data_driven,
    })

    ax.hist(values, bins=100, density=True, alpha=0.4, color="steelblue")
    ax.plot(grid, density, color="black", lw=1.2)
    ax.axvline(threshold, color="crimson", ls="--", lw=1.5,
               label=f"Threshold={threshold:.2f}\n(n={n_candidates}, {pct_of_total:.1f}%)")
    ax.axvline(bulk_x, color="gray", ls=":", lw=1, alpha=0.7)
    title = f"{factor}{direction}"
    if excluded_types:
        title += f"  [excluding {', '.join(sorted(excluded_types))}]"
    if is_manual:
        title += "  [manual threshold]"
    elif not is_data_driven:
        title += "  [fallback percentile]"
    ax.set_title(title)
    ax.set_xlabel("Factor value")
    ax.set_ylabel("Density" + (f" ({args.yscale})" if args.yscale != "linear" else ""))
    if args.yscale == "sqrt":
        ax.set_yscale("function", functions=(np.sqrt, np.square))
    else:
        ax.set_yscale(args.yscale)
    ax.legend(fontsize=8, loc="upper right" if direction == "+" else "upper left")

plt.tight_layout()
fig_suffix = f"_in_{celltype_suffix}" if celltype_suffix else ""
out_fig = os.path.join(args.output_dir, f"threshold_selection_histograms{fig_suffix}.png")
plt.savefig(out_fig, bbox_inches="tight", dpi=120)
plt.close("all")
print(f"\nSaved: {out_fig}")

pd.DataFrame(summary_rows).to_csv(
    os.path.join(args.output_dir, f"threshold_selection_summary{fig_suffix}.csv"), index=False
)
print(f"Saved: threshold_selection_summary{fig_suffix}.csv")

**Cell-type-filtered variants** (same factor, but considered only within specific cell types — reveals additional doublet populations that get lost in the global histogram):

In [ ]:
# DR21+ only within B cells (B/NK doublets)
args.factor_dirs = "DR21+"
args.celltype_filter = "B-cell"
args.exclude = ""
args.manual_threshold = ""
# ... re-run the cells above

In [ ]:
# DR35+ only within Monocytes-CD14 / Dendritic / Monocytes-CD16_FCGR3A (B/myeloid doublets)
args.factor_dirs = "DR35+"
args.celltype_filter = "Monocytes - CD14,Dendritic,Monocytes - CD16_FCGR3A"
# ... re-run the cells above

## 2. Candidate UMAP (factor-based)

*Source: `2_7_b_drvi_candidate_umap.py`*

DRVI candidate UMAP — shows the candidate cells selected by threshold
(2_7_a) on the DRVI UMAP, to visually check whether they form a compact,
distinct cluster (or overlap with an existing cluster).

Uses embed.h5ad (UMAP) and the candidate_cells_<factor><dir>.csv from 2_7_a.

Usage:
    conda run -n mapra_cytokines python 2_7_b_drvi_candidate_umap.py \
        --factor-dirs DR9-,DR17+,DR4+

In [ ]:
import argparse
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

In [ ]:
import argparse
args = argparse.Namespace(
    output_dir="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation",
    factor_dirs="DR9-,DR17+,DR4+,DR28+,DR21+",
    out_name="candidate_umap.png",
    celltype_key="cell_type_Scanorama",
    celltype_filter="",
    exclude="DR4+:Megakaryocytes",
)

In [ ]:
embed_path = os.path.join(args.output_dir, "embed.h5ad")
embed = sc.read_h5ad(embed_path)

if args.celltype_filter:
    keep = (embed.obs[args.celltype_key] == args.celltype_filter).values
    print(f"Filter '{args.celltype_key}' == '{args.celltype_filter}': {keep.sum()} / {embed.n_obs} cells")
    embed = embed[keep].copy()

umap = embed.obsm["X_umap"]
ct_suffix = f"_in_{args.celltype_filter.replace(' ', '_').replace('-', '')}" if args.celltype_filter else ""

exclude_map: dict[str, set[str]] = {}
for pair in [p.strip() for p in args.exclude.split(",") if p.strip()]:
    fd_key, celltype = pair.split(":")
    exclude_map.setdefault(fd_key.strip(), set()).add(celltype.strip())

factor_dirs = [fd.strip() for fd in args.factor_dirs.split(",") if fd.strip()]

sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
n = len(factor_dirs)
fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 5))
if n == 1:
    axes = [axes]

for ax, fd in zip(axes, factor_dirs):
    direction = fd[-1]
    factor = fd[:-1]
    safe = factor + direction.replace("+", "pos").replace("-", "neg")
    candidate_csv = os.path.join(args.output_dir, f"candidate_cells_{safe}{ct_suffix}.csv")
    if not os.path.exists(candidate_csv):
        print(f"  {fd}: {candidate_csv} not found — run 2_7_a first. Skipped.")
        continue

    candidates = set(pd.read_csv(candidate_csv)["cell_barcode"])
    is_candidate = embed.obs_names.isin(candidates)
    n_candidates = int(is_candidate.sum())

    excluded_types = exclude_map.get(fd, set())
    if excluded_types:
        keep_mask = ~embed.obs[args.celltype_key].isin(excluded_types).values
        print(f"  {fd}: excluding {int((~keep_mask).sum())} cells ({', '.join(sorted(excluded_types))})")
    else:
        keep_mask = np.ones(embed.n_obs, dtype=bool)

    plot_umap = umap[keep_mask]
    plot_candidate = is_candidate[keep_mask]

    ax.scatter(plot_umap[~plot_candidate, 0], plot_umap[~plot_candidate, 1],
               s=2, c="lightgray", alpha=0.5, linewidths=0, rasterized=True)
    ax.scatter(plot_umap[plot_candidate, 0], plot_umap[plot_candidate, 1],
               s=4, c="crimson", alpha=0.9, linewidths=0, rasterized=True)
    title = f"{fd}  (n={n_candidates}, {100*n_candidates/embed.n_obs:.2f}%)"
    if excluded_types:
        title += f"\n[excluding {', '.join(sorted(excluded_types))}]"
    ax.set_title(title)
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
out_name = args.out_name
if args.celltype_filter and args.out_name == "candidate_umap.png":
    out_name = f"candidate_umap{ct_suffix}.png"
out = os.path.join(args.output_dir, out_name)
plt.savefig(out, bbox_inches="tight", dpi=150)
plt.close("all")
print(f"Saved: {out}")

## 3. Gene threshold selection (analogous to 1., but on gene expression)

*Source: `2_7_c_drvi_gene_threshold_selection.py`*

Gene threshold selection — like 2_7_a, but on gene expression instead of DRVI
factors, and optionally filtered to a cell type (e.g. only B cells).

For each specified gene, the distribution of log1p-normalized expression
within the filtered cells is plotted as a histogram + KDE. As in 2_7_a, the
threshold is set in a data-driven way in the valley between the bulk peak
(no/low expression) and the target peak (high expression), with a fallback
to a percentile or manual override.

Result per gene:
  - Histogram plot with threshold line
  - candidate_cells_gene_<gene>_in_<celltype>.csv (cell barcodes above threshold)

Usage:
    conda run -n mapra_cytokines python 2_7_c_drvi_gene_threshold_selection.py \
        --genes IGHG3 --celltype-filter "B-cell"

In [ ]:
import argparse
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from scipy.signal import argrelextrema

In [ ]:
import argparse
args = argparse.Namespace(
    drvi_input="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/data/data_for_practicum_post_integration.h5ad",
    output_dir="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation",
    genes="IGHG3",
    celltype_key="cell_type_Scanorama",
    celltype_filter="B-cell",
    grid_points=2000,
    fallback_percentile=95.0,
    manual_threshold="IGHG3:0.2",
    yscale="sqrt",
)

In [ ]:
manual_threshold_map = {}
for pair in [p.strip() for p in args.manual_threshold.split(",") if p.strip()]:
    gene, value = pair.split(":")
    manual_threshold_map[gene.strip()] = float(value)

print("=== Load data ===")
adata = sc.read_h5ad(args.drvi_input)
adata.X = adata.layers["log1p_norm"]

if args.celltype_filter:
    keep_mask = (adata.obs[args.celltype_key] == args.celltype_filter).values
    print(f"Filter '{args.celltype_key}' == '{args.celltype_filter}': {keep_mask.sum()} / {adata.n_obs} cells")
    adata = adata[keep_mask].copy()
else:
    print(f"No cell-type filter — all {adata.n_obs} cells.")

genes = [g.strip() for g in args.genes.split(",") if g.strip()]
missing = [g for g in genes if g not in adata.var_names]
if missing:
    print(f"WARNING: not found in the dataset, skipped: {missing}")
genes = [g for g in genes if g in adata.var_names]
if not genes:
    raise RuntimeError("None of the specified genes were found in the dataset.")

n = len(genes)
fig, axes = plt.subplots(1, n, figsize=(6 * n, 4.5))
if n == 1:
    axes = [axes]

summary_rows = []

for ax, gene in zip(axes, genes):
    values = np.asarray(adata[:, gene].X.todense()).flatten() if hasattr(adata[:, gene].X, "todense") \
        else np.asarray(adata[:, gene].X).flatten()
    grid = np.linspace(values.min(), values.max(), args.grid_points)
    kde = gaussian_kde(values)
    density = kde(grid)

    maxima_idx = argrelextrema(density, np.greater)[0]
    minima_idx = argrelextrema(density, np.less)[0]
    bulk_idx = maxima_idx[np.argmax(density[maxima_idx])]
    bulk_x = grid[bulk_idx]

    # Gene expression: the target peak always lies to the right of the bulk peak (high expression).
    candidate_peaks = maxima_idx[grid[maxima_idx] > bulk_x]

    threshold = None
    is_data_driven = False
    is_manual = gene in manual_threshold_map

    if is_manual:
        threshold = manual_threshold_map[gene]
        is_data_driven = True
        print(f"{gene}: manual threshold={threshold} (overrides auto-detection)")
    elif len(candidate_peaks) > 0:
        target_idx = candidate_peaks[np.argmax(density[candidate_peaks])]
        lo, hi = sorted([bulk_idx, target_idx])
        valley_candidates = minima_idx[(minima_idx > lo) & (minima_idx < hi)]
        if len(valley_candidates) > 0:
            valley_idx = valley_candidates[np.argmin(density[valley_candidates])]
            threshold = grid[valley_idx]
            is_data_driven = True

    if threshold is None:
        threshold = np.percentile(values, args.fallback_percentile)

    candidate_mask = values > threshold
    n_candidates = int(candidate_mask.sum())

    candidates = pd.Series(adata.obs_names[candidate_mask], name="cell_barcode")
    ct_safe = args.celltype_filter.replace(" ", "_").replace("-", "") if args.celltype_filter else "all"
    out_csv = os.path.join(args.output_dir, f"candidate_cells_gene_{gene}_in_{ct_safe}.csv")
    candidates.to_csv(out_csv, index=False)

    # Donor breakdown: sample_id encodes patient.timepoint (e.g. "m6.4"),
    # patient_id = the part before the dot (same convention as in the
    # pseudobulk aggregation, see 2_2_b_drvi_pseudobulk.py).
    patient_id_all = adata.obs["sample_id"].astype(str).str.split(".", n=1).str[0]
    donor_counts = (
        patient_id_all[candidate_mask]
        .value_counts()
        .rename_axis("patient_id")
        .reset_index(name="n_candidate_cells")
    )
    donor_out_csv = os.path.join(args.output_dir, f"candidate_cells_gene_{gene}_in_{ct_safe}_by_donor.csv")
    donor_counts.to_csv(donor_out_csv, index=False)
    print(f"  Donors with candidate cells: {len(donor_counts)}  (saved: {os.path.basename(donor_out_csv)})")
    print(donor_counts.head(10).to_string(index=False))

    if is_manual:
        method = "manually set"
    elif is_data_driven:
        method = "data-driven (valley between peaks)"
    else:
        method = f"fallback: {args.fallback_percentile:.0f}th percentile"

    pct = 100 * n_candidates / adata.n_obs
    print(f"{gene}: threshold={threshold:.3f}  n_candidates={n_candidates} "
          f"({pct:.2f}% of {adata.n_obs} cells{' [' + args.celltype_filter + ']' if args.celltype_filter else ''})  [{method}]")
    print(f"  Saved: {os.path.basename(out_csv)}")

    summary_rows.append({
        "gene": gene, "celltype_filter": args.celltype_filter or "all",
        "threshold": threshold, "n_candidates": n_candidates, "pct_candidates": pct,
        "data_driven": is_data_driven,
    })

    ax.hist(values, bins=100, density=True, alpha=0.4, color="seagreen")
    ax.plot(grid, density, color="black", lw=1.2)
    ax.axvline(threshold, color="crimson", ls="--", lw=1.5,
               label=f"Threshold={threshold:.2f}\n(n={n_candidates}, {pct:.1f}%)")
    ax.axvline(bulk_x, color="gray", ls=":", lw=1, alpha=0.7)
    title = gene
    if args.celltype_filter:
        title += f"  [{args.celltype_filter}]"
    if is_manual:
        title += "  [manual threshold]"
    elif not is_data_driven:
        title += "  [fallback percentile]"
    ax.set_title(title)
    ax.set_xlabel("log1p-normalized expression")
    ax.set_ylabel("Density" + (f" ({args.yscale})" if args.yscale != "linear" else ""))
    if args.yscale == "sqrt":
        ax.set_yscale("function", functions=(np.sqrt, np.square))
    else:
        ax.set_yscale(args.yscale)
    ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
out_fig = os.path.join(args.output_dir, f"gene_threshold_selection_{ct_safe}.png")
plt.savefig(out_fig, bbox_inches="tight", dpi=120)
plt.close("all")
print(f"\nSaved: {out_fig}")

pd.DataFrame(summary_rows).to_csv(
    os.path.join(args.output_dir, f"gene_threshold_selection_summary_{ct_safe}.csv"), index=False
)
print(f"Saved: gene_threshold_selection_summary_{ct_safe}.csv")

Without a cell-type filter (across all cells), the algorithm usually finds a cleaner, genuinely data-driven bimodality, since the 'null peak' is much larger:

In [ ]:
args.celltype_filter = ""
args.manual_threshold = ""
# ... re-run the cells above

## 4. Candidate UMAP (gene-based, with majority-sample panel)

*Source: `2_7_d_drvi_gene_candidate_umap.py`*

Gene candidate UMAP — like 2_7_b, but for candidates from 2_7_c
(gene-based threshold selection, optionally filtered to a cell type).

Panel 1 shows the candidate cells on the full DRVI UMAP (all 109504 cells as
a gray background). Panel 2 (--add-majority-panel) zooms into the donor with
the most candidate cells: background = only that donor's own cells, so the
location of the signal within that single donor becomes visible without
being obscured by all other cells.

Usage:
    conda run -n mapra_cytokines python 2_7_d_drvi_gene_candidate_umap.py \
        --candidate-csv candidate_cells_gene_IGHG3_in_Bcell.csv \
        --title "IGHG3 (B-cell, threshold=1.37)" \
        --add-majority-panel

In [ ]:
import argparse
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt

In [ ]:
import argparse
args = argparse.Namespace(
    output_dir="/vol/disk/ubuntu/master_practicum_cytokines/repo/Mapra_Cytokines/visualization/drvi_interpretation",
    candidate_csv="candidate_cells_gene_IGHG3_in_Bcell.csv",
    title="IGHG3 in B-cell (threshold=1.37, 95th percentile fallback)",
    out_name="umap_IGHG3_Bcell_threshold1.37.png",
    add_majority_panel=True,
)

In [ ]:
embed_path = os.path.join(args.output_dir, "embed.h5ad")
embed = sc.read_h5ad(embed_path)
umap = embed.obsm["X_umap"]
patient_id = embed.obs["sample_id"].astype(str).str.split(".", n=1).str[0]

candidate_path = args.candidate_csv if os.path.isabs(args.candidate_csv) else os.path.join(args.output_dir, args.candidate_csv)
candidates = set(pd.read_csv(candidate_path)["cell_barcode"])
is_candidate = embed.obs_names.isin(candidates)
n_candidates = int(is_candidate.sum())

title = args.title or os.path.basename(candidate_path)

sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
n_panels = 2 if args.add_majority_panel else 1
fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 5.5))
axes = [axes] if n_panels == 1 else list(axes)

# Panel 1: all cells as background, all candidates in red.
ax = axes[0]
ax.scatter(umap[~is_candidate, 0], umap[~is_candidate, 1],
           s=2, c="lightgray", alpha=0.5, linewidths=0, rasterized=True)
ax.scatter(umap[is_candidate, 0], umap[is_candidate, 1],
           s=5, c="crimson", alpha=0.9, linewidths=0, rasterized=True)
ax.set_title(f"{title}\n(n={n_candidates}, {100*n_candidates/embed.n_obs:.2f}% of all cells)")
ax.set_xlabel("UMAP1")
ax.set_ylabel("UMAP2")
ax.set_xticks([])
ax.set_yticks([])

if args.add_majority_panel:
    donor_counts = patient_id[is_candidate].value_counts()
    majority_donor = donor_counts.idxmax()
    n_majority = int(donor_counts.iloc[0])

    donor_mask = (patient_id == majority_donor).values
    donor_candidate = is_candidate & donor_mask
    n_donor_cells = int(donor_mask.sum())

    ax2 = axes[1]
    ax2.scatter(umap[donor_mask & ~is_candidate, 0], umap[donor_mask & ~is_candidate, 1],
                s=4, c="lightgray", alpha=0.6, linewidths=0, rasterized=True)
    ax2.scatter(umap[donor_candidate, 0], umap[donor_candidate, 1],
                s=8, c="crimson", alpha=0.95, linewidths=0, rasterized=True)
    ax2.set_title(f"Donor {majority_donor} only (majority sample)\n"
                  f"(n={n_majority} candidates out of {n_donor_cells} cells from this donor, "
                  f"{100*n_majority/n_donor_cells:.1f}%)")
    ax2.set_xlabel("UMAP1")
    ax2.set_ylabel("UMAP2")
    ax2.set_xticks([])
    ax2.set_yticks([])
    print(f"Majority sample: {majority_donor}  ({n_majority}/{n_candidates} candidates, "
          f"{n_donor_cells} cells from this donor in total)")

plt.tight_layout()
out_name = args.out_name or f"umap_{os.path.splitext(os.path.basename(candidate_path))[0]}.png"
out = os.path.join(args.output_dir, out_name)
plt.savefig(out, bbox_inches="tight", dpi=150)
plt.close("all")
print(f"Saved: {out}  (n={n_candidates})")